### Agent Context

In [1]:
# 字典key是租户tenant_id，字典value是租户对应的user信息，也是字典。user字典的key是user_id
USER_DATABASE = {
    "school_a": {
        "u001": {"name": "虎哥", "course": "LangChain 开发指南"},
        "u002": {"name": "小明", "course": "Java开发入门"},
    },
    "school_b": {
        "u001": {"name": "涛哥", "course": "Python 入门"},
        "u002": {"name": "阿伟", "course": "VibeCoding 实战"},
    }
}

In [2]:
from dataclasses import dataclass

@dataclass(frozen=True)
class UserContext:
    """Agent运行时上下文"""
    user_id: str
    tenant_id: str


In [3]:
from langchain.tools import tool, ToolRuntime

@tool
def get_current_user_profile(runtime: ToolRuntime[UserContext]):
    """查询当前登录用户的资料。"""

    # 获取context
    ctx= runtime.context

    # 根据user_id和tenant_id获取用户信息
    profile = USER_DATABASE.get(ctx.tenant_id, {}).get(ctx.user_id)
    
    return (
        f"姓名：{profile['name']}，"
        f"课程：{profile['course']}，"
    )

In [5]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os

qwen_model = init_chat_model("openai:qwen3.7-plus", temperature=0, base_url=os.getenv("ROOT_URL"))

# 创建Agent时指定context_schema
agent = create_agent(
    model=qwen_model,
    tools=[get_current_user_profile],
    context_schema=UserContext,  # 指定Context类型
    system_prompt="你是课程平台助手。需要用户资料时调用工具。"
)

In [6]:
from langchain.messages import HumanMessage

# 调用时通过Context传递用户信息
response = agent.invoke(
    {"messages": [HumanMessage("Hello, 帮我查询我正在学习的课程")]},
    context=UserContext(user_id="u001", tenant_id="school_a")
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Hello, 帮我查询我正在学习的课程
================================== Ai Message ==================================
Tool Calls:
  get_current_user_profile (call_1fb0253a1a2b425eb0b1f823)
 Call ID: call_1fb0253a1a2b425eb0b1f823
  Args:
================================= Tool Message =================================
Name: get_current_user_profile

姓名：虎哥，课程：LangChain 开发指南，
================================== Ai Message ==================================

你好，虎哥！根据查询结果，你目前正在学习的课程是 **LangChain 开发指南**。

如果你需要关于这门课程的更多信息，比如课程进度、章节内容或其他帮助，请随时告诉我！


## Agent State
- messages
- other customized fields

In [7]:
from langchain.agents import AgentState


# 定义自定义State结构
class CustomState(AgentState):
    """Agent的任务状态"""
    model_call_count: int  # 模型调用次数

In [9]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage


@tool
def update_model_call_count(runtime: ToolRuntime):
    """A tool that count model call """

    # 获取state中的数据
    count = runtime.state.get("model_call_count", 0)
    print("model_call_count:", count)

    # 更新State中的数据
    update= {
        "model_call_count": count + 1, # 更新count值
        "messages": [ # 工具执行结束必须返回一条ToolMessage
            ToolMessage(
                content="Successfully updated agent state",
                tool_call_id=runtime.tool_call_id
            )
        ]
    }

    return Command(update=update)

In [10]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=qwen_model,
    tools=[update_model_call_count],
    state_schema=CustomState,  # 告诉Agent使用自定义的state
    checkpointer=InMemorySaver(),
    system_prompt="你是AI助手，你需要调用工具记录模型调用次数"
)

In [11]:
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [HumanMessage(content="Hi,我是虎哥")]},
    config
)

for message in response['messages']:
    message.pretty_print()
    
    
response = agent.invoke(
    {"messages": [HumanMessage(content="还记得我是谁吗")]},
    config
)

for message in response['messages']:
    message.pretty_print()

model_call_count: 0
model_call_count: 1
================================ Human Message =================================

Hi,我是虎哥
================================== Ai Message ==================================

你好虎哥！很高兴认识你。有什么我可以帮助你的吗？
Tool Calls:
  update_model_call_count (call_8179bd9862ef4bd6be9c80a1)
 Call ID: call_8179bd9862ef4bd6be9c80a1
  Args:
================================= Tool Message =================================
Name: update_model_call_count

Successfully updated agent state
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_8f0d14fb93744a76ab4c1f34)
 Call ID: call_8f0d14fb93744a76ab4c1f34
  Args:
================================= Tool Message =================================
Name: update_model_call_count

Successfully updated agent state
================================== Ai Message ==================================

你好虎哥！很高兴认识你。有什么我可以帮助你的吗？😊
model_call_count: 2
model_call_count: 3
model_ca

In [17]:
from langchain.tools import ToolRuntime, tool


# 定义方法，根据Context中的tenant_id和user_id拼接得到namespace
def preference_namespace(ctx: UserContext) -> tuple[str, ...]:
    return (
        "heima_platform",
        ctx.tenant_id,
        ctx.user_id
    )
# 定义固定的key
key = "learning_preferences"

@tool
def save_learning_preference(
    preference_name: str,
    preference_value: str,
    runtime: ToolRuntime[UserContext],
) -> str:
    """保存当前用户的长期学习偏好。"""

    # 通过上下文中的tenant_id和user_id来构建namespace
    namespace = preference_namespace(runtime.context)

    # 通过namespace和key来读取用户偏好, 返回结果item.value是dict格式
    item = runtime.store.get(namespace, key)
    preferences = dict(item.value) if item else {}

    # 记录本次要保存的用户偏好
    preferences[preference_name] = preference_value

    # 保存到Store中
    runtime.store.put(namespace, key, preferences)

    return f"已保存偏好：{preference_name}={preference_value}"


@tool
def get_learning_preferences(
    runtime: ToolRuntime[UserContext],
) -> str:
    """读取当前用户的长期学习偏好。"""

    namespace = preference_namespace(runtime.context)
    item = runtime.store.get(namespace, key)

    if item is None:
        return "当前用户还没有保存学习偏好"

    return f"当前学习偏好：{item.value}"

In [26]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore


checkpointer = InMemorySaver()
store = InMemoryStore()

agent = create_agent(
    model=qwen_model,
    tools=[
        get_current_user_profile,
        save_learning_preference,
        get_learning_preferences,
    ],
    context_schema=UserContext,
    checkpointer=checkpointer,
    store=store,
    system_prompt=(
        "你是课程平台助手。"
        "涉及用户资料时调用资料工具，"
        "涉及长期学习偏好或给用户讲解代码时调用偏好工具。"
    ),
)

In [27]:
context = UserContext(
    user_id="u001",
    tenant_id="school_a",
)

result = agent.invoke(
    {"messages":[HumanMessage(content="记住：我喜欢先看代码，再看原理")]},
    config={"configurable": {"thread_id": "thread-a"}},
    context=context,
)
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

记住：我喜欢先看代码，再看原理
================================== Ai Message ==================================
Tool Calls:
  save_learning_preference (call_b0ae271cf4bd47f79bbc50e4)
 Call ID: call_b0ae271cf4bd47f79bbc50e4
  Args:
    preference_name: learning_style
    preference_value: 先看代码，再看原理
================================= Tool Message =================================
Name: save_learning_preference

已保存偏好：learning_style=先看代码，再看原理
================================== Ai Message ==================================

好的，我已经记住了你的学习偏好：**先看代码，再看原理**。

以后在给你讲解代码或技术概念时，我会先展示代码示例，然后再解释背后的原理。这样可以帮助你更直观地理解内容。

如果你还有其他学习偏好想要保存，随时告诉我！


In [28]:
result = agent.invoke(
    {"messages":[HumanMessage(content="langchain如何实现结构化输出？")]},
    config={"configurable": {"thread_id": "thread-c"}},
    context=context,
)
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

langchain如何实现结构化输出？
================================== Ai Message ==================================
Tool Calls:
  get_learning_preferences (call_860e7520339e4b8592b8ec5c)
 Call ID: call_860e7520339e4b8592b8ec5c
  Args:
================================= Tool Message =================================
Name: get_learning_preferences

当前学习偏好：{'learning_style': '先看代码，再看原理'}
================================== Ai Message ==================================

根据你的学习偏好，我们先直接看最常用、最优雅的代码实现，然后再剖析它背后的原理。

### 💻 代码实现：使用 `with_structured_output`

在 LangChain 中，实现结构化输出最推荐的方式是结合 **Pydantic 模型** 和 **`with_structured_output`** 方法。

```python
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 1. 定义你期望的输出结构 (使用 Pydantic 定义 Schema)
class MovieReview(BaseModel):
    title: str = Field(description="电影名称")
    rating: int = Field(description="评分，1-10分")
    summary: str = Field(description="一句话总结")

# 